## Import

In [10]:
import collections
import pathlib

import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import layers, losses, utils
from tensorflow.keras.layers import TextVectorization
from pathlib import Path

import tensorflow_datasets as tfds
import tensorflow_text as tf_text

## Download and explore the dataset
- keras.utils.text_dataset_from_directory: for loading text-file examples.
- keras.layers.TextVectorization: for converting strings to token indices.

In [3]:
data_url = 'https://storage.googleapis.com/download.tensorflow.org/data/stack_overflow_16k.tar.gz'

dataset_dir = utils.get_file(
    origin=data_url,
    untar=True,
    cache_dir='stack_overflow',
    cache_subdir='')

dataset_dir = pathlib.Path(dataset_dir).parent

6053168/6053168 [==============================] - 3s 0us/step


In [4]:
list(dataset_dir.iterdir())

[WindowsPath('/tmp/.keras/README.md'),
 WindowsPath('/tmp/.keras/stack_overflow_16k.tar.gz'),
 WindowsPath('/tmp/.keras/test'),
 WindowsPath('/tmp/.keras/train')]

In [5]:
for path in dataset_dir.iterdir():
    print(path.absolute())

d:\tmp\.keras\README.md
d:\tmp\.keras\stack_overflow_16k.tar.gz
d:\tmp\.keras\test
d:\tmp\.keras\train


In [12]:
train_dir = Path('../tmp/.keras/train')
list(train_dir.iterdir())

[WindowsPath('../tmp/.keras/train/csharp'),
 WindowsPath('../tmp/.keras/train/java'),
 WindowsPath('../tmp/.keras/train/javascript'),
 WindowsPath('../tmp/.keras/train/python')]

In [15]:
sample_file = train_dir / 'python/0.txt'

with open(sample_file) as f:
    print(f.read())

"is it legal to define two methods with the same name but different returning types? i've written a piece of code to determine a typical palindrome string. i did this by the definition of a reverse() method returning a string. i also eager to have the same method, but in the void form, because of some future needs..as i add the latter to the code, the valid output will become invalid..so, the question is that is it legal to define two methods with the same name but different returning types?.if not, please let me know how to write this code with the void-type method...class detector(object):.    def __init__(self,string):.        self.string = string..    forbidden = (' ','!','?','.','-','_','&amp;','%',""#"","","")..    def eliminator(self):.        for item in self.forbidden:.            if item in self.string:.                self.string = self.string.replace(item,"""")..    def reverse(self):.        return self.string[::-1]            ..    #def reverse(self):.    #    self.string

## Training Set

In [ ]:
batch_size = 32
seed = 42

"""โค๊ดชุดนี้บอกว่า เราจะเอาข้อมูลจาก train_dir มาใช้สำหรับการเทรนโมเดล 80 เปอร์เซนต์ คำว่า subset คือบอกว่า
ข้อมูลที่เอามา จะอยู่ใน subset ที่ชื่อว่า train """
raw_train_ds = utils.text_dataset_from_directory(
    train_dir,
    batch_size=batch_size,
    validation_split=0.2,
    subset='training',
    seed=seed
)

Found 8000 files belonging to 4 classes.
Using 6400 files for training.


In [ ]:
"""take(1): หมายความว่า เอา batch แรกออกมาแสดงผล"""
i = 0

for text_batch, label_batch in raw_train_ds.take(1):
    for i in range(10):
        print("Question: ", text_batch.numpy()[i])
        print("Label: ", label_batch.numpy()[i])

Question:  b'"blank8 why is my solution faster than the neat solution? (hackerrank chocolate feast) edit: simplified my solution..edit: removed opinion based secondary question...background: atarted learning blank a week or two ago using hackerranks problems as exercises and stackoverflow search + google as my teacher, i\'ve had some limited experience learning other languages...i did the exercise my own ""noobish learner way"" which i can\'t help but feel is a ""botched job"" when i see ""neat &amp; short"" solutions...however, when submitting both solutions one after another a couple of times i found the ""neat"" solution was quite a bit slower. ..i vaguely remember something about % operations being costly, is mine faster because of no % operations or is there more to it than just that?..exercise: https://www.hackerrank.com/challenges/chocolate-feast..neat solution from discussion:..import blank.io.*;.import blank.util.*;..public class solution {.    static int cc; .    public stati

In [19]:
"""โขว์ Label ทั้งหมดที่มีใน dataset"""
i = 0

for i, label in enumerate(raw_train_ds.class_names):
    print("Label", i, "corresponds to", label)

Label 0 corresponds to csharp
Label 1 corresponds to java
Label 2 corresponds to javascript
Label 3 corresponds to python


## Validation Set

In [20]:
raw_val_ds = utils.text_dataset_from_directory(
    train_dir,
    batch_size=batch_size,
    validation_split=0.2,
    subset='validation',
    seed=seed
)

Found 8000 files belonging to 4 classes.
Using 1600 files for validation.


## Test Set

In [21]:
test_dir = Path('../tmp/.keras/test')

raw_test_ds = utils.text_dataset_from_directory(
    test_dir,
    batch_size=batch_size
)

Found 8000 files belonging to 4 classes.


In [24]:
i = 0
for text_batch, label_batch in raw_test_ds.take(1):
    for i in range(3):
        print("Label:", label_batch.numpy()[i],"Question:", text_batch.numpy()[i])

Label: 1 Question: b'"speed variation of a parachutist in my exercice, i need to calculate the speed\'s variation of a parachutist using some mathematical equation given in exercise. i enter the mass = 80kg and the height = 39000 m..i have to display this message on the sreen :.system.out.printf(""%.0f, %.4f, %.4f, %.5fn"",.                    t, height, speed, accel);..normally, i should have this result :..&gt; 133, 1991.2751, 284.9225, -79.22827.&gt; 134, 1742.1436, 216.8788, -57.96464.&gt; 135, 1551.4499, 167.0971, -42.40784.&gt; 136, 1403.5103, 130.6760, -31.02624...but unfortunately, this is what i have on my screen :..&lt; 133, 1991.2751, 284.9224, -79.22826.&lt; 134, 1742.1436, 216.8788, -57.96463.&lt; 135, 1551.4499, 167.0971, -42.40783.&lt; 136, 1403.5103, 130.6759, -31.02623...this is my code here :..    final double g = 9.8100;.    float surface = 2;.    int tf = 171;.    double v0 = 00.0;.    double t0 = 00.0;..    for (double t = 0; t &lt; tf; t++) .    {.        double h

## Prepare data for training
    TextVectorization คือ Layer สำหรับเตรียมข้อความให้พร้อมใช้งานในโมเดล TensorFlow หรือพูดง่าย ๆ ก็คือ:
    เอาข้อความ (Text) → แปลงเป็นตัวเลข (Numbers) เพื่อให้โมเดลเข้าใจและนำไปใช้เรียนรู้ต่อได้

    หน้าที่หลักๆ ของ TextVectorization มีดังนี้ 
    - การจัดคำ หมายถึง การลบเครื่องหมายต่างๆ และองค์ประกอบของ HTML ออกไปให้เหลือแต่ Text
    - Tokenization หมายถึง การแยกสตริงออกเป็น Token เช่น จากประโยคให้กลายเป็น คำ ซึ่งแต่ละคำอาจจะแทนด้วย Token
    - Vectoriztation หมายถึง การแปลง Token เหล่านั้นเป็นตัวเลขเฉพาะเพื่อให้ โมเดล เรียนรู้ได้

    
    parameters's TextVectorization
    tf.keras.layers.TextVectorization(
    max_tokens=None,
    standardize='lower_and_strip_punctuation',
    split='whitespace',
    ngrams=None,
    output_mode='int',
    output_sequence_length=None,
    pad_to_max_tokens=False,
    vocabulary=None,
    idf_weights=None,
    sparse=False,
    ragged=False,
    encoding='utf-8',
    name=None,
    **kwargs
    )

In [ ]:
VOCAB_SIZE = 10000
MAX_SEQUENCE_LENGTH = 250

int_vectorize_layer = TextVectorization(
    max_tokens=VOCAB_SIZE,
    standardize='lower_and_split_punct',
    split='whitespace',
    output_mode='int',
    output_sequence_length=MAX_SEQUENCE_LENGTH
)

In [ ]:
"""เราแปลงคำเป็นตัวเลขและ โดยใช้ TextVectorization layer ต่อมาคือ คำที่เราแปลงอะ 
โมเดลยังไม่รู้จักคำเหล่านั้นเลยว่าจะใช้งานยังไง เราเลยต้องการที่จะใช้ .adapt() เพื่อให้โมเดลรู้จักคำเหล่านั้นก่อนล่วงหน้า
ก่อนเอาไปประมวลผล"""

train_text = raw_train_ds.map(lambda text, label: text)
int_vectorize_layer.adapt(train_text)